In [1]:
%cd ../

/Users/emir.demic/Desktop/PhD/Research/Irena 2025/moral-reasoning-2026a


# Imports

In [2]:
import typing as t
import collections

import pandas as pd
import numpy as np

from src import util
from src.descriptives import descriptive_stat, scale_stat

# Missing Data Handling

In [3]:
df = pd.read_csv("data/raw/01_raw_data_all.csv", na_values=["", " "])
df.head()

/var/folders/2v/k185_q9n7nvg8j7820jcmw1c0000gp/T/ipykernel_41227/1379845968.py:1: DtypeWarning: Columns (76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/raw/01_raw_data_all.csv", na_values=["", " "])


,order,Participant_ID,Vignette_ID,attcheck,age,gender,education,residence,ses,MFQ2_C_1,...,AOT,CM,PIT,PE,MFQ_C,MFQ_E,MFQ_PRO,MFQ_L,MFQ_A,MFQ_P
0,5625,724,CA1,1.0,49.0,1,5,4,3,5,...,NaN,NaN,NaN,NaN,4.5,4.666667,4.5,3.833333,3.166667,2.0
1,6020,724,CA2,1.0,49.0,1,5,4,3,5,...,NaN,NaN,NaN,NaN,4.5,4.666667,4.5,3.833333,3.166667,2.0
2,6415,724,CA3,1.0,49.0,1,5,4,3,5,...,NaN,NaN,NaN,NaN,4.5,4.666667,4.5,3.833333,3.166667,2.0
3,6810,724,LO1,1.0,49.0,1,5,4,3,5,...,NaN,NaN,NaN,NaN,4.5,4.666667,4.5,3.833333,3.166667,2.0
4,7205,724,LO2,1.0,49.0,1,5,4,3,5,...,NaN,NaN,NaN,NaN,4.5,4.666667,4.5,3.833333,3.166667,2.0


In [4]:
to_drop = [
    "wrongness",
    "process_raw",
    "process_code",
    "certainty",
    "foundation_code",
    "notes",
    "attcheck",
    "order",
    "Vignette_ID",
    "value_relevance",
    "value_relevance_cont"
]   

short_format_df = df[[i for i in df.columns if i not in to_drop]].drop_duplicates(
    subset=["Participant_ID"], keep="first"
)
short_format_df.head()

,Participant_ID,age,gender,education,residence,ses,MFQ2_C_1,MFQ2_C_2,MFQ2_C_3,MFQ2_C_4,...,AOT,CM,PIT,PE,MFQ_C,MFQ_E,MFQ_PRO,MFQ_L,MFQ_A,MFQ_P
0,724,49.0,1,5,4,3,5,5,3,4,...,NaN,NaN,NaN,NaN,4.500000,4.666667,4.500000,3.833333,3.166667,2.000000
18,727,38.0,1,4,3,3,5,4,5,5,...,1.000000,7.0,7.000000,6.000000,4.666667,4.333333,5.000000,3.500000,4.000000,4.000000
36,88,30.0,1,4,4,2,3,2,3,2,...,1.166667,5.0,6.166667,4.833333,2.333333,3.666667,3.833333,2.333333,3.166667,3.166667
54,544,51.0,1,6,1,4,5,5,3,5,...,1.166667,6.5,6.500000,5.166667,4.666667,4.166667,5.000000,4.000000,4.500000,4.166667
72,115,50.0,1,3,1,1,5,5,5,5,...,1.333333,6.0,6.500000,6.166667,5.000000,4.833333,4.666667,3.666667,2.666667,4.000000


In [5]:
short_format_df.shape

(766, 95)

In [6]:
ids_to_drop = set()
for i in short_format_df.columns:
    if i.lower().startswith("crt"):
        continue
    ids_to_drop.update(
        short_format_df.loc[short_format_df[i].isna(), "Participant_ID"].to_list()
    )
    missing = short_format_df[i].isna().sum()
    if missing > 0:
        print(f"Variable {i} has {missing} instances of missing observations.")

Variable age has 1 instances of missing observations.
Variable AOT_1r has 1 instances of missing observations.
Variable AOT_2r has 1 instances of missing observations.
Variable AOT_3r has 1 instances of missing observations.
Variable AOT_4r has 1 instances of missing observations.
Variable AOT_5r has 1 instances of missing observations.
Variable AOT_6r has 1 instances of missing observations.
Variable CMT_1 has 1 instances of missing observations.
Variable CMT_2 has 1 instances of missing observations.
Variable CMT_3 has 1 instances of missing observations.
Variable CMT_4 has 1 instances of missing observations.
Variable CMT_5 has 1 instances of missing observations.
Variable CMT_6 has 1 instances of missing observations.
Variable PIT_1 has 1 instances of missing observations.
Variable PIT_2 has 1 instances of missing observations.
Variable PIT_3 has 1 instances of missing observations.
Variable PIT_4 has 1 instances of missing observations.
Variable PIT_5 has 1 instances of missing ob

In [7]:
# Participants younger than 18
short_format_df.loc[short_format_df["age"] < 18, "Participant_ID"].to_list()

[428, 426]

In [8]:
# Participants older than 80
short_format_df.loc[short_format_df["age"] > 80, "Participant_ID"].to_list()

[346]

In [9]:
ids_to_drop.update([428, 426, 346])
print(ids_to_drop)

{361, 426, 428, 724, 346}


In [10]:
short_format_df = short_format_df.loc[~short_format_df["Participant_ID"].isin(ids_to_drop), :]

In [11]:
df["wrongness"].value_counts()

wrongness
5    6070
3    2250
4    1970
1     938
2     931
0     134
Name: count, dtype: int64

In [12]:
df["certainty"].value_counts()

certainty
7    6922
6    2495
5    1572
4     772
3     222
0     134
1      97
2      79
Name: count, dtype: int64

In [13]:
df = df.loc[~df["Participant_ID"].isin(ids_to_drop), :]

### Create Long Format DF

In [14]:
long_format_df = df[
    [
        "order",
        "Participant_ID",
        "Vignette_ID",
        "wrongness",
        "certainty",
        "process_raw",
        "process_code",
        "foundation_code",
        "notes",
        "value_relevance",
        "value_relevance_cont",
        "CRT_Correct",
        "CRT_Heuristic",
        "CRT_Incorrect",
        "AOT",
        "CM",
        "PIT",
        "PE",
        "MFQ_C",
        "MFQ_E",
        "MFQ_PRO",
        "MFQ_L",
        "MFQ_A",
        "MFQ_P",
    ]
]

In [15]:
long_format_df = long_format_df[long_format_df["Vignette_ID"] != "0"]

In [16]:
for i in ["CRT1", "CRT2", "CRT3", "CRT4", "CRT5", "CRT6", "CRT7", "CRT8"]:
    short_format_df[i + "_correct"] = short_format_df[i].replace(2, 0)

In [18]:
long_format_df["process_code"].replace({"i": "I", "r": "R", "a": "A"}, inplace=True)
long_format_df["process_code"].value_counts()

process_code
I    6853
R    4013
A    1223
Name: count, dtype: int64

# Write dataframes

In [19]:
long_format_df.to_csv("data/processed/01_long_form_data.csv", index=False)
short_format_df.to_csv("data/processed/01_short_form_data.csv", index=False)